# Hillstrom E????????????????????

## ?????????

??????????E?????????????????????????????????????12?????????64,000?????????3??????????????:

- 1/3???:???????E????????????
- 1/3???:???????E????????????
- 1/3???:E?????????(???)

## ?????

### ????(???)

| ??? | ?? |
|--------|------|
| **Recency** | ???????????? |
| **History_Segment** | ??1?????????????? |
| **History** | ??1??????????(??) |
| **Mens** | ??1??????????????????(1=?????0=???????) |
| **Womens** | ??1??????????????????(1=?????0=???????) |
| **Zip_Code** | ???????(??????????) |
| **Newbie** | ????????(1=??12????????0=????) |
| **Channel** | ??1?????????????? |
| **Segment** | ????????E????????????(?Mens E-Mail???Womens E-Mail???No E-Mail?) |

### ??/????(???????)

| ??? | ?? |
|--------|------|
| **treatment** | E????????????????????<br>???1: E???????????(?Mens E-Mail?????Womens E-Mail?)<br>???0: E??????????????(?No E-Mail?) |
| **group** | ???/??????????<br>?'treatment': E?????????????(???/???)<br>?'control': E????????????????(???) |

### ????(????)

| ??? | ?? |
|--------|------|
| **Visit** | E??????2????????????????????(1=?????0=???????) |
| **Conversion** | E??????2????????????????(1=?????0=???????) |
| **Spend** | E??????2????????????(??) |

## ?????

???????????????????????????????????????????????????????:

1. E?????????????????(?????????)?????????
2. ?????E??????????E??????????
3. ????????????E????????????????(???????)???
4. ??????????????????????????????????????????????

## ???????

??????????????????????:

- ???????????????????(ATE)???
- ??????????????????(CATE)???
- ?????????????????????????????????????

In [ ]:
# Uplift Modeling API????????????Notebook

import requests
import json
import pandas as pd
API_URL = "https://causal-chart-api-620283975862.asia-northeast1.run.app"

# 1. ??????????
train_payload = {
    "data": [
        {"Recency": 10, "History": 100, "Mens": 1, "Womens": 0, "Newbie": 0, "treatment": 1, "Conversion": 1},
        {"Recency": 12, "History": 120, "Mens": 1, "Womens": 0, "Newbie": 0, "treatment": 1, "Conversion": 0},
        {"Recency": 8, "History": 90, "Mens": 1, "Womens": 0, "Newbie": 0, "treatment": 0, "Conversion": 0},
        {"Recency": 15, "History": 80, "Mens": 1, "Womens": 0, "Newbie": 0, "treatment": 0, "Conversion": 1},
        {"Recency": 5, "History": 200, "Mens": 0, "Womens": 1, "Newbie": 0, "treatment": 0, "Conversion": 0},
        {"Recency": 7, "History": 180, "Mens": 0, "Womens": 1, "Newbie": 0, "treatment": 0, "Conversion": 1},
        {"Recency": 3, "History": 210, "Mens": 0, "Womens": 1, "Newbie": 0, "treatment": 1, "Conversion": 0},
        {"Recency": 4, "History": 220, "Mens": 0, "Womens": 1, "Newbie": 0, "treatment": 1, "Conversion": 1},
        {"Recency": 20, "History": 50, "Mens": 0, "Womens": 0, "Newbie": 1, "treatment": 0, "Conversion": 0},
        {"Recency": 25, "History": 30, "Mens": 0, "Womens": 0, "Newbie": 1, "treatment": 1, "Conversion": 1}
    ],
    "features": ["Recency", "History", "Mens", "Womens", "Newbie"],
    "treatment_col": "treatment",
    "outcome_col": "Conversion",
    "model_type": "s_learner",
    "model_params": {
        "n_estimators": 100,
        "max_depth": 5
    }
}

response = requests.post(f"{API_URL}/train", json=train_payload)
train_result = response.json()
print("Train Result:", json.dumps(train_result, indent=2, ensure_ascii=False))


In [ ]:
train_result["model_path"]

In [ ]:
# ?????????????????
model_path = train_result["model_path"]

# 2. ?????
predict_payload = {
    "features": [
        {"Recency": 8, "History": 150, "Mens": 1, "Womens": 0, "Newbie": 1},
        {"Recency": 3, "History": 250, "Mens": 0, "Womens": 1, "Newbie": 0}
    ],
    "model_path": model_path
}

response = requests.post(f"{API_URL}/predict", json=predict_payload)
predict_result = response.json()
print("Predict Result:", json.dumps(predict_result, indent=2, ensure_ascii=False))



"""

curl -X POST "https://causal-chart-api-620283975862.asia-northeast1.run.app" \
  -H "Content-Type: application/json" \
  -d '{
    "features": [
      {"Recency": 8, "History": 150, "Mens": 1, "Womens": 0, "Newbie": 1},
      {"Recency": 3, "History": 250, "Mens": 0, "Womens": 1, "Newbie": 0}
    ],
    "model_path": "s_learner_20250302_085207.pkl"
  }'
"""

In [ ]:
import pandas as pd
import numpy as np
import requests
import json
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split



# ???????????????
def calculate_uplift_metrics(y_true, treatment, uplift_scores, n_bins=10):
    """????????????????????????"""
    # ?????????
    df = pd.DataFrame({
        'y': y_true,
        'w': treatment,
        'uplift': uplift_scores
    })
    
    # ????????????n_bins?????????
    df['bin'] = pd.qcut(df['uplift'], q=n_bins, labels=False, duplicates='drop')
    
    # ??????
    bin_stats = []
    total_treat = df[df['w'] == 1].shape[0]
    total_control = df[df['w'] == 0].shape[0]
    total_treat_converted = df[(df['w'] == 1) & (df['y'] == 1)].shape[0]
    total_control_converted = df[(df['w'] == 0) & (df['y'] == 1)].shape[0]
    
    for i in range(n_bins):
        bin_df = df[df['bin'] == i]
        if len(bin_df) == 0:
            continue
            
        # ?????????????????????????
        treat_size = bin_df[bin_df['w'] == 1].shape[0]
        control_size = bin_df[bin_df['w'] == 0].shape[0]
        
        if treat_size == 0 or control_size == 0:
            continue
            
        treat_conv = bin_df[(bin_df['w'] == 1) & (bin_df['y'] == 1)].shape[0]
        control_conv = bin_df[(bin_df['w'] == 0) & (bin_df['y'] == 1)].shape[0]
            
        # ???????????????
        treat_conv_rate = treat_conv / treat_size
        control_conv_rate = control_conv / control_size
        bin_uplift = treat_conv_rate - control_conv_rate
        
        bin_stats.append({
            'bin': i,
            'size': bin_df.shape[0],
            'treat_size': treat_size,
            'control_size': control_size,
            'treat_conv': treat_conv,
            'control_conv': control_conv,
            'treat_conv_rate': treat_conv_rate,
            'control_conv_rate': control_conv_rate,
            'uplift': bin_uplift,
            'mean_pred_uplift': bin_df['uplift'].mean()
        })
    
    # ?????????????????
    if not bin_stats:
        return {
            'auuc': 0.0,
            'qini': 0.0,
            'uplift_at_top_k': 0.0,
            'corr': 0.0
        }
    
    bin_df = pd.DataFrame(bin_stats)
    bin_df = bin_df.sort_values('mean_pred_uplift', ascending=False)
    
    # ??????????
    # AUUC?Qini??
    total_pop = total_treat + total_control
    cum_treat = 0
    cum_control = 0
    cum_treat_conv = 0
    cum_control_conv = 0
    
    auuc_values = []
    random_auuc_values = []
    
    for _, row in bin_df.iterrows():
        cum_treat += row['treat_size']
        cum_control += row['control_size']
        cum_treat_conv += row['treat_conv']
        cum_control_conv += row['control_conv']
        
        # ????????
        cum_treat_rate = cum_treat_conv / cum_treat if cum_treat > 0 else 0
        cum_control_rate = cum_control_conv / cum_control if cum_control > 0 else 0
        
        # ?????????
        overall_treat_rate = total_treat_converted / total_treat if total_treat > 0 else 0
        overall_control_rate = total_control_converted / total_control if total_control > 0 else 0
        
        auuc_values.append((cum_treat_rate - cum_control_rate) * (cum_treat + cum_control) / total_pop)
        random_auuc_values.append((overall_treat_rate - overall_control_rate) * (cum_treat + cum_control) / total_pop)
    
    # AUUC
    auuc = np.trapz(auuc_values) if auuc_values else 0
    random_auuc = np.trapz(random_auuc_values) if random_auuc_values else 0
    
    # Qini
    qini = (auuc - random_auuc) / (abs(random_auuc) + 1e-10) if random_auuc != 0 else 0
    
    # ??20%????????
    top_20_pct = max(1, int(len(bin_df) * 0.2))
    top_k_uplift = bin_df.iloc[:top_20_pct]['uplift'].mean() if len(bin_df) > 0 else 0
    
    # ??
    corr = bin_df['mean_pred_uplift'].corr(bin_df['uplift']) if len(bin_df) > 1 else 0
    
    return {
        'auuc': float(auuc),
        'qini': float(qini),
        'uplift_at_top_k': float(top_k_uplift),
        'corr': float(corr)
    }

def plot_uplift_curve(y_true, treatment, uplift_scores, title="Uplift Curve", save_path=None):
    """??????????????????"""
    # ?????????
    df = pd.DataFrame({
        'y': y_true,
        'w': treatment,
        'uplift': uplift_scores
    })
    
    # ??????????
    df = df.sort_values('uplift', ascending=False)
    
    # ?????????????????
    n_bins = 10
    population_fractions = np.linspace(0, 1, n_bins+1)[1:]
    
    incremental_uplifts = []
    random_uplifts = []
    
    # ???????????
    total_treat_rate = df[df['w'] == 1]['y'].mean()
    total_control_rate = df[df['w'] == 0]['y'].mean()
    total_uplift = total_treat_rate - total_control_rate
    
    for fraction in population_fractions:
        subset_size = int(len(df) * fraction)
        subset = df.iloc[:subset_size]
        
        # ????????????????
        treat_rate = subset[subset['w'] == 1]['y'].mean() if subset[subset['w'] == 1].shape[0] > 0 else 0
        control_rate = subset[subset['w'] == 0]['y'].mean() if subset[subset['w'] == 0].shape[0] > 0 else 0
        
        incremental_uplifts.append(treat_rate - control_rate)
        random_uplifts.append(total_uplift * fraction)
    
    # ??????
    fig, ax = plt.subplots(figsize=(10, 6))
    
    ax.plot(population_fractions, incremental_uplifts, marker='o', label='Model Uplift')
    ax.plot(population_fractions, random_uplifts, linestyle='--', label='Random Targeting')
    ax.axhline(y=total_uplift, color='r', linestyle=':', label=f'Average Uplift ({total_uplift:.5f})')
    
    ax.set_xlabel('Population Fraction')
    ax.set_ylabel('Uplift')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # ???????????????
    metrics = calculate_uplift_metrics(y_true, treatment, uplift_scores)
    metrics_text = f"AUUC: {metrics['auuc']:.5f}\nQini: {metrics['qini']:.5f}\nTop 20% Uplift: {metrics['uplift_at_top_k']:.5f}"
    ax.text(0.02, 0.02, metrics_text, transform=ax.transAxes, 
            bbox=dict(boxstyle='round,pad=0.5', facecolor='white', alpha=0.7))
    
    if save_path is not None:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    
    return fig

def main():
    # 1. ???????
    try:
        df = pd.read_csv("hillstrom.csv")
    except FileNotFoundError:
        print("hillstrom.csv???????????????????????????????????????")
        return
    
    print(f"?????: {len(df)}")
    
    # 2. ???
    df.columns = [col.capitalize() for col in df.columns]
    df['treatment'] = (df['Segment'] != 'No E-Mail').astype(int)
    
    # ????????
    features = ['Recency', 'History', 'Mens', 'Womens', 'Newbie']
    
    # ????? (?????????????????)
    train_df, test_df = train_test_split(df, test_size=0.3, random_state=42, stratify=df['treatment'])
    print(f"?????: {len(train_df)}?, ??????: {len(test_df)}?")
    
    # 3. ??????????????API?????
    train_data = train_df.to_dict('records')
    
    # ??????????????
    model_types = ["s_learner", "t_learner", "x_learner", "r_learner", "causal_tree", "uplift_rf"]
    model_paths = {}
    
    for model_type in model_types:
        print(f"\n??????: {model_type}")
        train_payload = {
            "data": train_data,
            "features": features,
            "treatment_col": "treatment",
            "outcome_col": "Conversion",
            "model_type": model_type,
            "model_params": {
                "n_estimators": 100,
                "max_depth": 5
            }
        }
        
        response = requests.post(f"{API_URL}/train", json=train_payload)
        if response.status_code == 200:
            result = response.json()
            model_path = result["model_path"]
            model_paths[model_type] = model_path
            metrics = result["model_info"]["metrics"]
            print(f"  ??: {model_type}")
            print(f"  ??ATE: {metrics.get('actual_ate'):.5f}")
            print(f"  ??ATE: {metrics.get('estimated_ate'):.5f}")
        else:
            print(f"  ??: {response.text}")
    
    # 4. ?????????????
    test_features = test_df[features].to_dict('records')
    
    # 5. ??????????
    predictions = {}
    
    for model_type, model_path in model_paths.items():
        print(f"\n??: {model_type}")
        predict_payload = {
            "features": test_features,
            "model_path": model_path
        }
        
        try:
            response = requests.post(f"{API_URL}/predict", json=predict_payload)
            response.raise_for_status()  # ?????????4xx/5xx?????????
            result = response.json()
            preds = result["predictions"]
            print(f"  ??: {len(preds)}????")
            
            # ??????????
            if len(preds) != len(test_df):
                print(f"  ??: ?????({len(preds)})??????????({len(test_df)})??????")
                if len(preds) == 2 * len(test_df):
                    # causal_tree?1????????2???????????
                    preds = preds[::2]
                    print(f"  2????????1??????: {len(preds)}?")
                elif len(preds) > len(test_df):
                    preds = preds[:len(test_df)]
                    print(f"  ?????????????: {len(preds)}?")
                else:
                    mean_val = np.mean(preds)
                    preds = preds + [mean_val] * (len(test_df) - len(preds))
                    print(f"  ???????????????: {len(preds)}?")
            
            # R-learner??????
            if model_type == 'r_learner':
                preds_array = np.array(preds, dtype=float)
                outliers = np.abs(preds_array) > 1
                if np.any(outliers):
                    outlier_count = np.sum(outliers)
                    print(f"  ??: {outlier_count}????????")
                    median = np.median(preds_array[~outliers]) if np.sum(~outliers) > 0 else 0
                    preds_array[outliers] = median
                    print(f"  ???????({median:.5f})???")
                    preds = preds_array.tolist()
            
            predictions[model_type] = preds
        except Exception as e:
            print(f"  ??: {str(e)}")
    
    # 6. ????????
    y_test = test_df['Conversion'].values
    w_test = test_df['treatment'].values
    
    print("\n?????????????:")
    all_scores = {}
    
    for model_type, preds in predictions.items():
        try:
            # NaN?????????
            preds_array = np.array(preds, dtype=float)
            if np.isnan(preds_array).any():
                print(f"  {model_type}: NaN?????????????????")
                mean_val = np.nanmean(preds_array)
                preds_array = np.nan_to_num(preds_array, nan=mean_val)
            
            # ????????
            metrics = calculate_uplift_metrics(y_test, w_test, preds_array)
            all_scores[model_type] = metrics['qini']
            
            print(f"  {model_type}:")
            print(f"    Qini??: {metrics['qini']:.4f}")
            print(f"    AUUC: {metrics['auuc']:.4f}")
            print(f"    ??20%??????: {metrics['uplift_at_top_k']:.4f}")
            print(f"    ??-????: {metrics['corr']:.4f}")
            
            # ??????????????
            save_path = f"uplift_curve_{model_type}.png"
            fig = plot_uplift_curve(
                y_test, w_test, preds_array,
                title=f"Uplift Curve - {model_type}",
                save_path=save_path
            )
            print(f"  ?????????? '{save_path}' ?????????")
            plt.close(fig)
            
        except Exception as e:
            print(f"  {model_type}??????: {str(e)}")
    
    # 7. ?????
    if all_scores:
        print("\n=== ??????? ===")
        sorted_models = sorted(all_scores.items(), key=lambda x: x[1], reverse=True)
        for i, (model, score) in enumerate(sorted_models):
            print(f"{i+1}?: {model} (Qini??: {score:.4f})")
        
        best_model = sorted_models[0][0]
        print(f"\n?????: {best_model} (Qini??: {all_scores[best_model]:.4f})")
        
        # ???????????
        plt.figure(figsize=(12, 8))
        
        for model_type, preds in predictions.items():
            # ?????
            df = pd.DataFrame({
                'y': y_test,
                'w': w_test,
                'uplift': np.array(preds, dtype=float)
            })
            
            # ??????????
            df = df.sort_values('uplift', ascending=False)
            
            # ?????????????????
            population_fractions = np.linspace(0, 1, 21)[1:]  # 5%??
            incremental_uplifts = []
            
            for fraction in population_fractions:
                subset_size = int(len(df) * fraction)
                subset = df.iloc[:subset_size]
                
                treat_size = subset[subset['w'] == 1].shape[0]
                control_size = subset[subset['w'] == 0].shape[0]
                
                if treat_size > 0 and control_size > 0:
                    treat_rate = subset[subset['w'] == 1]['y'].mean()
                    control_rate = subset[subset['w'] == 0]['y'].mean()
                    incremental_uplifts.append(treat_rate - control_rate)
                else:
                    # ?????????????????????????0
                    incremental_uplifts.append(incremental_uplifts[-1] if incremental_uplifts else 0)
            
            # ????
            plt.plot(population_fractions, incremental_uplifts, marker='o', 
                     label=f"{model_type} (Qini={all_scores[model_type]:.4f})")
        
        # ????????????
        total_treat_rate = df[df['w'] == 1]['y'].mean()
        total_control_rate = df[df['w'] == 0]['y'].mean()
        total_uplift = total_treat_rate - total_control_rate
        
        plt.plot(population_fractions, [total_uplift * f for f in population_fractions], 
                 linestyle='--', label='Random Targeting')
        plt.axhline(y=total_uplift, color='r', linestyle=':', 
                   label=f'Average Uplift ({total_uplift:.5f})')
        
        plt.xlabel('Population Fraction')
        plt.ylabel('Uplift')
        plt.title('Uplift Curves - Model Comparison')
        plt.legend(loc='best')
        plt.grid(True, alpha=0.3)
        plt.show()
        model_comparison_path = "uplift_model_comparison.png"
        #plt.savefig(model_comparison_path, dpi=300, bbox_inches='tight')
        #print(f"\n?????????? '{model_comparison_path}' ?????????")
        #plt.close()


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        import traceback
        print(f"??????????: {str(e)}")
        print(traceback.format_exc())


In [ ]:

df = pd.read_csv("../../hillstrom.csv")
print("?????:", len(df))
df.head()


In [ ]:
pd.DataFrame(train_payload["data"])